# Python for Finance — Session 3 in-class practice

Two  mock-exam exercises from Sessions 1 and 2. \
The goal is to reconstruct a solution from the problem, not to copy and paste from the slides. Write a short plan before coding (meta-algorithm).


## Self-assessment

> **Level 0 — Memory:** begin without resources.  
> **Level 1 — Slides:** consult Session-1/Session-2 slides if a concept is familiar but syntax is missing.  
> **Level 2 — Hints:** after a serious attempt, use the hint appendix at the end of this notebook.  
> **Level 3 — Previous notebooks:** if still blocked, consult the earlier companion notebook. This is permitted and useful, but it diagnoses something you cannot yet reconstruct independently and should revise before the closed-book/no-AI exam.

A serious attempt means that you have identified inputs and outputs, written at least a partial algorithm, and tried a first implementation or isolated the step that blocks you.\
**AI assistance is not part of this exercise workflow.**


## Exercise A — Expense claims under a simple policy

An organisation receives nine expense claims.\
Each position across the five lists describes one claim. 
A claim is assessed by the following internal policy:

1. An amount that is zero or negative is rejected.
2. A category not present in `category_limits` is rejected.
3. A receipt is required when the claimed amount is greater than EUR 25; \
without it, the claim is rejected.
4. Otherwise, the reimbursed amount is the smaller of the claim and the category limit.
5. A valid claim below or at the limit is labelled `approved`; a valid claim reduced to the limit is labelled `capped`; all others are `rejected`.

Produce a claim-level audit trail and an employee-level reimbursement summary.  

The order of the input claims must be preserved.


In [1]:
claim_ids = ["C101", "C102", "C103", "C104", "C105",
             "C106", "C107", "C108", "C109"]

employees = ["Amina", "Leo", "Amina", "Marta", "Leo",
             "Marta", "Amina", "Leo", "Marta"]

categories = ["train", "meal", "hotel", "taxi", "hotel",
              "meal", "meal", "train", "hotel"]

amounts = [145.0, 42.0, 260.0, 30.0, 180.0,
           -5.0, 18.0, 210.0, 130.0]

has_receipt = [True, True, True, True, False,
               True, False, True, True]

category_limits = {
    "train": 160.0,
    "meal": 35.0,
    "hotel": 200.0,
}


### Tasks

1. **Check the inputs.**\
State and check the assumptions that must hold before the lists can be treated as aligned claim records.\
At minimum, all five lists must have the same length and claim identifiers must be unique.
2. **Separate one decision from the whole report.**\
Define a function that assesses one category/amount/receipt combination under the supplied limits and returns both a status and a reimbursed amount. 
The function must implement the rule order above (not print).
3. **Build the audit trail.**\
Process all claims and create `claim_results`, a list of dictionaries with exactly these keys: `claim_id`, `employee`, `status`, `reimbursed`. 
Preserve the original claim order.
4. **Summarise by employee.**\
Create `totals_by_employee` (total reimbursed amount per employee) and `rejected_by_employee` (number of rejected claims per employee).
5. **Answer the management questions.**\
Identify the employee with the largest reimbursement total and construct `review_ids`, the ordered claim IDs whose status is either `capped` or `rejected`.
6. **Audit the result (write a py code to audit your result).**\
Check programmatically that every input claim produced exactly one result, no reimbursement is negative, and the sum of employee totals equals the sum of claim-level reimbursements.


### Your meta-algorithm

Write the stages and the intermediate objects you want before translating into Python.


In [2]:
# TODO: write your plan as comments.


### A1. Input checks


In [ ]:
# 


### A2. Audit trail and employee summaries


In [ ]:
# 


### A3. Management questions and non-revealing checks


In [5]:
# TODO: identify the largest-total employee, build review_ids, and audit your result.

exercise_a_complete = False  # change to True when the required objects exist
if exercise_a_complete:
    assert len(claim_results) == len(claim_ids)
    assert all(set(result) == {"claim_id", "employee", "status", "reimbursed"}
               for result in claim_results)
    assert all(result["reimbursed"] >= 0 for result in claim_results)
    assert sum(totals_by_employee.values()) == sum(
        result["reimbursed"] for result in claim_results
    )


### A4. Self-auditing

In [ ]:
EXPECTED_RESULTS = [
    ("C101", "Amina", "approved", 145.0),
    ("C102", "Leo", "capped", 35.0),
    ("C103", "Amina", "capped", 200.0),
    ("C104", "Marta", "rejected", 0.0),
    ("C105", "Leo", "rejected", 0.0),
    ("C106", "Marta", "rejected", 0.0),
    ("C107", "Amina", "approved", 18.0),
    ("C108", "Leo", "capped", 160.0),
    ("C109", "Marta", "approved", 130.0),
]
EXPECTED_TOTALS = {"Amina": 363.0, "Leo": 195.0, "Marta": 130.0}
EXPECTED_REJECTED = {"Amina": 0, "Leo": 1, "Marta": 2}
EXPECTED_REVIEW = ["C102", "C103", "C104", "C105", "C106", "C108"]

# check (ASSERT) against your result

### Self-diagnosis — Exercise A

How much do you rate yourself?

- [ ] 0 — solved from memory
- [ ] 1 — consulted slides
- [ ] 2 — used one or more hints
- [ ] 3 — consulted a previous companion notebook


## Exercise B — Parcel-delivery reliability by region

An operations manager wants to compare delivery reliability across two regions.\
A row records the number of parcels and late parcels for one date, depot, and service.\
Reliability means the fraction delivered on time:

`on-time rate = (recorded parcels - late parcels) / recorded parcels`.

Rows without a recorded parcel total cannot contribute to this rate.\
Report regional performance and show how express-service reliability evolved day by day. The manager's reference level is 92%.


In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("data/session3_parcel_reliability.csv")

depots = pd.DataFrame({
    "depot_id": ["D01", "D02", "D03", "D04"],
    "region": ["North", "North", "South", "South"],
})

assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"


### Tasks

1. **Inspect before calculating.** Load the local file and report its dimensions, column names, first rows, and inferred dtypes. Identify why `parcels` is not numeric.
2. **Repair the analytical types.** Make `date` a date-like column and represent `not_recorded` parcel totals as missing numeric values. Report the missing count and create an analysis table containing only rows with a recorded parcel denominator. Do not replace missing parcel totals by zero.
3. **Attach region and audit the match.** Add the region from the supplied depot table. Check that this operation preserves the 218 analysis rows and that every depot found a region.
4. **Construct the quantity needed for the rate.** Add one column containing on-time parcels. Check that it is never negative and never exceeds total parcels.
5. **Produce the regional service table.** For each region/service pair, report recorded parcels, late parcels, on-time parcels, number of contributing rows, and the weighted on-time rate. State the best and worst region/service combinations.
6. **Answer the daily express question.** Keep express-service rows, produce one daily on-time rate per region, and arrange the result as dates × regions.
7. **Audit the analytical matrix.** Check its shape, missingness, finite values, and range; count for each region how many days fall below 92%.
8. **Communicate the result.** Draw one figure with date on the x-axis and on-time rate on the y-axis, one line per region, a legend, informative labels/title, and a horizontal 92% reference line. In one or two sentences, answer the manager's question.


### Your Meta-algorithm

Start from the required numerator and denominator. Decide which table should exist after repair, after the regional match, after aggregation, and immediately before plotting.


In [ ]:
# TODO:


### B1. Inspect and repair


In [ ]:
# 


### B2. Attach region and construct on-time parcels


In [ ]:
# 


### B3. Regional/service and daily express evidence


In [ ]:
# 


### B4. Matrix audit, threshold counts and figure


In [11]:
# TODO: audit the matrix, count below-reference days, plot, and interpret.

exercise_b_complete = False  # change to True when the required objects exist
if exercise_b_complete:
    values = daily_express.to_numpy()
    assert daily_express.shape == (28, 2)
    assert not daily_express.isna().any().any()
    assert np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all()


(**Interpretation:** Write one or two sentences here after producing the figure.)


### Self-diagnosis — Exercise B

What was the highest support level I needed?

- [ ] 0 — solved from memory
- [ ] 1 — consulted slides
- [ ] 2 — used one or more hints
- [ ] 3 — consulted a previous companion notebook


---------------

## Hint appendix — open only after a serious attempt

Read only as far as you need.


-----------------

### Exercise A — Hint 1: organise the problem before coding

- Apply the policy rules in the correct order. First decide whether a claim must be rejected; only then consider whether an otherwise valid claim needs to be capped.

- Break the task into three separate steps: assess one claim, process all claims, and summarise the results by employee.

- Think about the form of each final output. Claim-level results should remain in the original claim order, while employee summaries are naturally stored using the employee name as a key.

*If this is enough to restart your work, stop here before reading the next hint.*

### Exercise A — Hint 2: decide what information to store

- Assessing one claim should give you two outputs: its status (approved, capped, or rejected) and its reimbursable amount.

- For each processed claim, store four pieces of information: the claim ID, employee, status, and reimbursable amount.

- Before you can identify the employee with the largest reimbursement, you first need the total reimbursable amount for each employee.

- Also keep track of the number of rejected claims for each employee. This is a separate summary from the reimbursement totals.

*If this is enough to restart your work, stop here before reading the next hint.*


### Exercise A — Hint 3: Python tools that may help

- zip() can be used to move through the aligned claim ID, employee, category, amount, and receipt lists at the same time.

- A dictionary can store running totals or counts by employee. When an employee appears for the first time, initialise the corresponding total or count before updating it.

- A loop or list comprehension can collect the IDs of claims whose status is capped or rejected.

- assert statements can be used to check that the inputs have compatible lengths and that independently computed totals agree.

- If you need a syntax reminder, revisit the Session 1 material on zip(), functions, dictionaries, list comprehensions, and assertions.


-----------

### Exercise B — Hint 1: organise the problem before coding

- Before computing a reliability rate, make sure that both the numerator and denominator are usable. Rows with an unavailable parcel total cannot be used for the rate calculation.

- Regional reliability must reflect parcel volume. Therefore, sum the on-time parcels and total parcels within each group first, then divide. Do not average the row-level percentages.

- For the final plot, you need one reliability value for each date-region combination, rather than one line for every raw depot observation.

*If this is enough to restart your work, stop here before reading the next hint.*


### Exercise B — Hint 2: decide what intermediate results you need

- After cleaning the data, the analysis table should contain 218 rows, with the parcel-total column stored as numeric values and the date column stored as dates.

- After merging the region lookup, every remaining row should have a region and the number of rows should still be 218.

- Before calculating reliability, create the total number of parcels and total number of on-time parcels for each group.

- The final table used for the Express plot should contain 28 dates as rows and 2 regions as columns.

*If this is enough to restart your work, stop here before reading the next hint.*


### Exercise B — Hint 3: Python tools that may help

- Use an appropriate conversion step to turn the parcel-total column into numeric data and the date column into a proper date type. Values that cannot be converted can be represented as missing.

- Use a merge to attach the region lookup to the depot observations. Check the merge carefully to confirm that each depot maps to the intended region and that no unexpected rows are added or lost.

- Use groupby() with the relevant grouping variables to sum the on-time parcels and total parcels before computing the reliability rate.

- Use a reshaping operation such as pivot() or pivot_table() to turn the daily Express results into a table with dates in the rows and regions in the columns.

- Once the final table is small, you can inspect its shape and values directly and use simple checks for missing values, valid ranges, and the number of observations below the 92% reference level.

- If you need a syntax reminder, revisit the Session 2 material on “When numbers are read as text,” “Auditing a merge,” “Split, apply, combine,” and “From long to wide.”
